### INSTALACIONES E IMPORTACIONES

In [1]:
pip install pyedflib #Recomendado para ver eeg en python

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#Recomendado'

[notice] A new release of pip available: 22.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd

In [4]:
import pyedflib


In [5]:
import mne

In [6]:
import pooch

In [7]:
import numpy as np
import matplotlib.pyplot as plt

In [8]:
import antropy as ant

In [9]:
import matplotlib.pyplot as plt

In [10]:
import zipfile


In [11]:
import os

In [12]:
import pathlib

In [53]:
from scipy.stats import mannwhitneyu

En esta pagina https://physionet.org/content/eegmat/1.0.0/ vamos a encontrar eeg realizado a un grupo de sujetos
en modo reposo y durante la resolución de ejercicios matemáticos. Uno por uno analizaremos la dimensión fractal de cada área del cerebro. Luego crearemos df para comparar las variaciones en general de todos los sujetos según áreas del cerebro. Tenemos sujeto00_01 es el sujeto 00 en estado reposo y sujeto00_02 el mismo pero resolviendo ejercicios matematicos.

Hay 21 canales EEG (Fp1, Fp2, F3, F4, etc.).
La frecuencia de muestreo es 500 Hz (500 puntos por segundo por canal).
Está filtrado de 0.5 Hz a 45 Hz.
El registro dura ~182 segundos (~3 minutos).

Frontal: Fp1, Fp2, F3, F4, F7, F8
Temporal: T3, T4, T5, T6
Parietal: P3, P4, Pz
Occipital: O1, O2
Central: C3, C4, Cz

### Calculo DM por tramo entero de los primeros 9 sujetos


In [13]:
url = "https://physionet.org/static/published-projects/eegmat/1.0.0/Subject00_1.edf"

# Descargar temporalmente
path = pooch.retrieve(
    url=url,
    known_hash=None  # No verificamos hash para agilizar
)

# Abrir con MNE
raw = mne.io.read_raw_edf(path, preload=True)
print(raw.info)

Extracting EDF parameters from C:\Users\yamal\AppData\Local\pooch\pooch\Cache\53c52e9eb313b49f18dcacb950744ce1-Subject00_1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 90999  =      0.000 ...   181.998 secs...
<Info | 8 non-empty values
 bads: []
 ch_names: EEG Fp1, EEG Fp2, EEG F3, EEG F4, EEG F7, EEG F8, EEG T3, EEG ...
 chs: 21 EEG
 custom_ref_applied: False
 highpass: 0.5 Hz
 lowpass: 45.0 Hz
 meas_date: 2011-01-01 00:00:00 UTC
 nchan: 21
 projs: []
 sfreq: 500.0 Hz
 subject_info: <subject_info | his_id: 0, sex: 1, last_name: Subject0, birthday: 1990-01-01>
>


In [14]:

# Archivo donde guardaremos todo
csv_file = "resultados_fractales.csv"

# Lista de sujetos (puedes ajustar el rango)
sujetos = [f"Subject{i:02d}" for i in range(10)]  # 00 a 09

# Lista para acumular resultados
resultados = []

for sujeto in sujetos:
    for estado in [1, 2]:  # 1 = reposo, 2 = actividad
        condicion = "reposo" if estado == 1 else "actividad"
        
        # URL EDF
        url = f"https://physionet.org/static/published-projects/eegmat/1.0.0/{sujeto}_{estado}.edf"
        print(f"Procesando: {sujeto} - {condicion}")

        # Descargar
        path = pooch.retrieve(url=url, known_hash=None)

        # Leer EDF
        raw = mne.io.read_raw_edf(path, preload=True)
        data, _ = raw.get_data(return_times=True)
        canales = raw.ch_names

        # Calcular Higuchi FD para cada canal
        for i, canal in enumerate(canales):
            valor_fd = ant.higuchi_fd(data[i], kmax=10)
            resultados.append({
                "sujeto": sujeto[-2:],      # "00", "01", ...
                "condicion": condicion,     # "reposo" o "actividad"
                "canal": canal,             # Nombre del canal
                "valor_fractal": valor_fd   # FD calculada
            })

# Crear DataFrame long
df_long = pd.DataFrame(resultados)

# Guardar CSV
df_long.to_csv(csv_file, index=False)

print("\n✅ Listo. Archivo guardado en:", csv_file)
print(df_long.head())


Procesando: Subject00 - reposo
Extracting EDF parameters from C:\Users\yamal\AppData\Local\pooch\pooch\Cache\53c52e9eb313b49f18dcacb950744ce1-Subject00_1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 90999  =      0.000 ...   181.998 secs...
Procesando: Subject00 - actividad
Extracting EDF parameters from C:\Users\yamal\AppData\Local\pooch\pooch\Cache\497838b58060f20164e2cf29584f519f-Subject00_2.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 30999  =      0.000 ...    61.998 secs...
Procesando: Subject01 - reposo
Extracting EDF parameters from C:\Users\yamal\AppData\Local\pooch\pooch\Cache\6f1180d28763a7ade30931f2400ef8cd-Subject01_1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 90999  =      0.000 ...   181.998 secs...
Procesando: Subject01 - actividad
Extracting EDF parameters from C:\Users\yamal\AppData\Local\poo

In [15]:
df_long.head()

,sujeto,condicion,canal,valor_fractal
0,00,reposo,EEG Fp1,1.102250
1,00,reposo,EEG Fp2,1.119314
2,00,reposo,EEG F3,1.101865
3,00,reposo,EEG F4,1.097340
4,00,reposo,EEG F7,1.124143


In [16]:
# Convertir de formato long a wide por sujeto y canal
df_var = df_long.pivot_table(
    index=["sujeto", "canal"],     # cada combinación de sujeto y canal es una fila
    columns="condicion",           # columnas separadas: reposo, actividad
    values="valor_fractal",
    aggfunc="first"                # usamos 'first' porque no hay duplicados
).reset_index()

# Calcular variación
df_var["variacion"] = df_var["actividad"] - df_var["reposo"]


In [17]:
df_var.head()

condicion,sujeto,canal,actividad,reposo,variacion
0,00,ECG ECG,1.073343,1.059201,0.014142
1,00,EEG A2-A1,1.159064,1.083710,0.075354
2,00,EEG C3,1.109658,1.103591,0.006068
3,00,EEG C4,1.107483,1.101031,0.006452
4,00,EEG Cz,1.103878,1.099786,0.004093


In [18]:
df_var.nlargest(10, "variacion", keep="all")

condicion,sujeto,canal,actividad,reposo,variacion
145,06,EEG T5,1.283424,1.168686,0.114738
141,06,EEG P4,1.209026,1.103526,0.105500
144,06,EEG T4,1.203839,1.109055,0.094784
142,06,EEG Pz,1.210310,1.117937,0.092372
140,06,EEG P3,1.230099,1.140035,0.090064
21,01,ECG ECG,1.240052,1.150989,0.089063
146,06,EEG T6,1.191778,1.104541,0.087236
106,05,EEG A2-A1,1.203858,1.118286,0.085572
143,06,EEG T3,1.222456,1.136941,0.085515
129,06,EEG C4,1.198444,1.114210,0.084233


In [19]:
df_var.nsmallest(10, "variacion", keep="all")

condicion,sujeto,canal,actividad,reposo,variacion
169,08,EEG A2-A1,1.105634,1.178122,-0.072488
84,04,ECG ECG,1.104135,1.160495,-0.056360
42,02,ECG ECG,1.198266,1.246566,-0.048300
40,01,EEG T5,1.141489,1.183206,-0.041717
38,01,EEG T3,1.190670,1.230370,-0.039700
178,08,EEG Fp2,1.144509,1.179140,-0.034631
196,09,EEG F7,1.158128,1.184147,-0.026018
48,02,EEG F4,1.189255,1.214668,-0.025413
87,04,EEG C4,1.095301,1.118798,-0.023497
94,04,EEG Fp2,1.104804,1.125741,-0.020937


### Veo el csv

In [20]:
info_sujetos = pd.read_csv("C:\\Users\\yamal\\OneDrive\\Escritorio\\Proyectos en github\\Fractals-dimension-eeg\\eeg-during-mental-arithmetic-tasks-1.0.0\\eeg-during-mental-arithmetic-tasks-1.0.0\\subject-info.csv")


In [21]:
info_sujetos.head()

,Subject,Age,Gender,Recording year,Number of subtractions,Count quality
0,Subject00,21,F,2011,9.70,0
1,Subject01,18,F,2011,29.35,1
2,Subject02,19,F,2012,12.88,1
3,Subject03,17,F,2010,31.00,1
4,Subject04,17,F,2010,8.60,0


In [22]:
info_sujetos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Subject                 36 non-null     object 
 1   Age                     36 non-null     int64  
 2   Gender                  36 non-null     object 
 3   Recording year          36 non-null     int64  
 4   Number of subtractions  36 non-null     float64
 5   Count quality           36 non-null     int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 1.8+ KB


In [23]:
info_sujetos.describe()

,Age,Recording year,Number of subtractions,Count quality
count,36.000000,36.000000,36.000000,36.000000
mean,18.250000,2010.611111,17.612222,0.722222
std,2.169595,0.802773,9.694809,0.454257
min,16.000000,2010.000000,1.000000,0.000000
25%,17.000000,2010.000000,9.925000,0.000000
50%,17.000000,2010.000000,16.000000,1.000000
75%,19.000000,2011.000000,26.520000,1.000000
max,26.000000,2012.000000,34.590000,1.000000


In [24]:
info_sujetos.drop("Recording year",axis=1, inplace=True)

In [25]:
info_sujetos.head()

,Subject,Age,Gender,Number of subtractions,Count quality
0,Subject00,21,F,9.70,0
1,Subject01,18,F,29.35,1
2,Subject02,19,F,12.88,1
3,Subject03,17,F,31.00,1
4,Subject04,17,F,8.60,0


In [26]:
info_sujetos.groupby("Count quality").count()

,Subject,Age,Gender,Number of subtractions
Count quality,,,,
0,10,10,10,10
1,26,26,26,26


In [27]:
info_sujetos.groupby(["Gender", "Count quality"]).size()

Gender  Count quality
F       0                 7
        1                20
M       0                 3
        1                 6
dtype: int64

### DF fraccionado

In [28]:
def leer_edf(path):
    raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
    data = raw.get_data()
    chans = raw.ch_names
    sfreq = raw.info['sfreq']
    return data, chans, sfreq

In [29]:
"""
Recibe: información de un canal.
Retorna: La información del canal dividida en 9 segmentos de 20 segundos
"""


def dividir20(data , duracion, sfreq):
    duracion_bloque = 20  # segundos
    muestras_bloque = int(sfreq * duracion_bloque)


    bloques_20 = []
    for i in range(int(duracion/duracion_bloque)):
        start = i * muestras_bloque
        end = start + muestras_bloque
        bloque = data[start:end]
        bloques_20.append(bloque)

    return bloques_20

In [30]:
"""
Recorro cada canal y a cada uno le aplico la función de dividirlo en bloques de 20 segundos.
Crea un diccionario de listas cuya clave es cada canal y sus valores los bloques temporales.
"""

def DividirPorCanal(data_sujeto, sfreq, chans):
    diccionario_canales = {}
    for canal, data_canal in enumerate(data_sujeto):
        division_canal = dividir20(data_canal, data_sujeto.shape[1]/sfreq, sfreq)
        diccionario_canales[chans[canal]]=division_canal

    return diccionario_canales

In [31]:
"""
En la libreria Antropy existe la funcion de Higuchi para df, usaré esa misma
Hago una funcion que calcula la df por canal y por bloque de 20 seg.

Tengo 9 dimensiones fractales por canal, voy a sacar el promedio.
hicimos 9 bloques de 20 segundos para que el resultado sea mas acertado. Calcular dm en bloques 
muy grandes de tiempo no es recomendable
"""

def FDPorCanal (diccionario_canales):
    diccionario_df = {}
    diccionario_promedio = {}
    for canal in diccionario_canales:
        listaDF_canal = []
        for i in diccionario_canales[canal]:
           fd_higuchi= ant.higuchi_fd(i, kmax = 10)
           listaDF_canal.append(fd_higuchi)
        diccionario_df[canal] = listaDF_canal.copy()
        diccionario_promedio[canal] = sum(listaDF_canal)/len(listaDF_canal)


    return diccionario_promedio

In [32]:
#Hago todo el proceso de calcular df en una sola función para aplicar por .edf. l uego automatizo
def CalculoDF(path):
    data, chans, sfreq = leer_edf(path)
    diccionario_canales = DividirPorCanal(data, sfreq, chans)
    diccionario_promedio = FDPorCanal(diccionario_canales)

    return diccionario_promedio

In [33]:
CalculoDF("C:\\Users\\yamal\\OneDrive\\Escritorio\\Proyectos en github\\Fractals-dimension-eeg\\eeg-during-mental-arithmetic-tasks-1.0.0\\eeg-during-mental-arithmetic-tasks-1.0.0\\Subject00_1.edf")

{'EEG Fp1': 1.1026697167219437,
 'EEG Fp2': 1.1197299952555118,
 'EEG F3': 1.1023168695391117,
 'EEG F4': 1.097645296425881,
 'EEG F7': 1.1245940331192672,
 'EEG F8': 1.123973039934177,
 'EEG T3': 1.106592869123511,
 'EEG T4': 1.0939193767175963,
 'EEG C3': 1.1040681865520763,
 'EEG C4': 1.1014770461326568,
 'EEG T5': 1.0963556405576853,
 'EEG T6': 1.0844444649099594,
 'EEG P3': 1.0928653997554731,
 'EEG P4': 1.0813178565008126,
 'EEG O1': 1.088543072829842,
 'EEG O2': 1.0696603841278245,
 'EEG Fz': 1.0970443525125064,
 'EEG Cz': 1.1001175981758557,
 'EEG Pz': 1.0915019734261662,
 'EEG A2-A1': 1.0846257754778854,
 'ECG ECG': 1.059151792969644}

In [34]:
mainPath = "C:\\Users\\yamal\\OneDrive\\Escritorio\\Proyectos en github\\Fractals-dimension-eeg\\eeg-during-mental-arithmetic-tasks-1.0.0\\eeg-during-mental-arithmetic-tasks-1.0.0"
result = []
for file in os.listdir(mainPath):
    if file.startswith("Subject") and file.endswith(".edf"):
        filePath = os.path.join(mainPath, file)
        df_Subject = CalculoDF(filePath)
        df_Subject["Sujeto"] = file
        result.append(df_Subject)
df_resultado = pd.DataFrame(result)
df_resultado.set_index("Sujeto", inplace=True)
print (df_resultado.head(2))

                  EEG Fp1  EEG Fp2    EEG F3    EEG F4    EEG F7    EEG F8  \
Sujeto                                                                       
Subject00_1.edf  1.102670  1.11973  1.102317  1.097645  1.124594  1.123973   
Subject00_2.edf  1.134732  1.13094  1.111707  1.105077  1.144263  1.147398   

                   EEG T3    EEG T4    EEG C3    EEG C4  ...    EEG T6  \
Sujeto                                                   ...             
Subject00_1.edf  1.106593  1.093919  1.104068  1.101477  ...  1.084444   
Subject00_2.edf  1.124691  1.112482  1.110506  1.108315  ...  1.105374   

                   EEG P3    EEG P4    EEG O1    EEG O2    EEG Fz    EEG Cz  \
Sujeto                                                                        
Subject00_1.edf  1.092865  1.081318  1.088543  1.069660  1.097044  1.100118   
Subject00_2.edf  1.115240  1.104324  1.101014  1.081384  1.103726  1.104388   

                   EEG Pz  EEG A2-A1   ECG ECG  
Sujeto                  

In [35]:
#Para que no corte al imprimir el df
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
print(df_resultado)


                  EEG Fp1   EEG Fp2    EEG F3    EEG F4    EEG F7    EEG F8  \
Sujeto                                                                        
Subject00_1.edf  1.102670  1.119730  1.102317  1.097645  1.124594  1.123973   
Subject00_2.edf  1.134732  1.130940  1.111707  1.105077  1.144263  1.147398   
Subject01_1.edf  1.145034  1.146002  1.140149  1.126669  1.151928  1.140875   
Subject01_2.edf  1.175808  1.168565  1.136131  1.140059  1.156185  1.160347   
Subject02_1.edf  1.265691  1.170144  1.165206  1.215721  1.260586  1.244903   
Subject02_2.edf  1.272918  1.214251  1.194197  1.189885  1.273985  1.239024   
Subject03_1.edf  1.123835  1.117084  1.088036  1.087552  1.116321  1.101685   
Subject03_2.edf  1.147961  1.154362  1.107989  1.103213  1.124487  1.119959   
Subject04_1.edf  1.113540  1.124073  1.110060  1.110950  1.110502  1.118236   
Subject04_2.edf  1.114715  1.105356  1.102425  1.098530  1.117504  1.105082   
Subject05_1.edf  1.134807  1.173265  1.108969  1.103

#Ahora voy a calcular la media y dispersión de los sujetos en reposo por canal y en resolución de actividades. Esto es para tener una idea general de la variación por canales. Sin embargo, la hipótesis principal es que la dimensión fractal aumenta en aquellos que han resuelto erradamente.

In [36]:
canales = list(df_resultado.columns)
print(canales)

['EEG Fp1', 'EEG Fp2', 'EEG F3', 'EEG F4', 'EEG F7', 'EEG F8', 'EEG T3', 'EEG T4', 'EEG C3', 'EEG C4', 'EEG T5', 'EEG T6', 'EEG P3', 'EEG P4', 'EEG O1', 'EEG O2', 'EEG Fz', 'EEG Cz', 'EEG Pz', 'EEG A2-A1', 'ECG ECG']


In [37]:
df_reposo = df_resultado[df_resultado.index.str.contains("_1.edf")]

In [38]:
df_actividad = df_resultado[df_resultado.index.str.contains("_2.edf")]


In [39]:
df_reposo.describe()

,EEG Fp1,EEG Fp2,EEG F3,EEG F4,EEG F7,EEG F8,EEG T3,EEG T4,EEG C3,EEG C4,EEG T5,EEG T6,EEG P3,EEG P4,EEG O1,EEG O2,EEG Fz,EEG Cz,EEG Pz,EEG A2-A1,ECG ECG
count,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000
mean,1.146541,1.149342,1.121862,1.122196,1.146112,1.148531,1.124842,1.127626,1.113956,1.113839,1.129027,1.116404,1.102669,1.096264,1.097767,1.095207,1.116638,1.113664,1.097541,1.170200,1.117698
std,0.040343,0.037576,0.024976,0.029352,0.035507,0.034198,0.031551,0.029291,0.024515,0.028235,0.035292,0.037564,0.031222,0.026556,0.036517,0.035976,0.022374,0.024309,0.027406,0.052025,0.045866
min,1.086274,1.098322,1.088036,1.082540,1.070995,1.101685,1.081216,1.078448,1.064400,1.065206,1.075844,1.062258,1.049534,1.050086,1.050287,1.050334,1.081047,1.062874,1.050171,1.084626,1.059152
25%,1.118140,1.122987,1.104135,1.098454,1.122520,1.123950,1.101763,1.107798,1.098812,1.098012,1.098531,1.092866,1.080323,1.076599,1.072074,1.070626,1.100479,1.098749,1.078463,1.123316,1.089735
50%,1.137810,1.141506,1.115199,1.118271,1.144558,1.141049,1.119417,1.117942,1.111264,1.109187,1.125541,1.112784,1.099231,1.095207,1.090133,1.084935,1.115700,1.114675,1.094857,1.165867,1.107993
75%,1.160145,1.173361,1.138046,1.137370,1.162930,1.162754,1.138181,1.147805,1.128065,1.122192,1.160636,1.135213,1.117946,1.108668,1.110964,1.109316,1.130728,1.126497,1.113013,1.201318,1.133667
max,1.265691,1.265191,1.169292,1.215721,1.260586,1.244903,1.214111,1.189013,1.165595,1.215356,1.208108,1.254588,1.197425,1.159312,1.200352,1.220105,1.160649,1.166565,1.156333,1.291428,1.247263


In [40]:
df_actividad.describe()

,EEG Fp1,EEG Fp2,EEG F3,EEG F4,EEG F7,EEG F8,EEG T3,EEG T4,EEG C3,EEG C4,EEG T5,EEG T6,EEG P3,EEG P4,EEG O1,EEG O2,EEG Fz,EEG Cz,EEG Pz,EEG A2-A1,ECG ECG
count,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000
mean,1.174114,1.175159,1.141843,1.141225,1.169069,1.171090,1.151721,1.157941,1.132269,1.133515,1.157084,1.142073,1.125014,1.120516,1.123479,1.117273,1.133156,1.131589,1.122395,1.200897,1.126096
std,0.046707,0.049568,0.036709,0.036371,0.042458,0.049234,0.044288,0.052213,0.034153,0.038410,0.048200,0.049274,0.038938,0.038301,0.049502,0.046338,0.030505,0.032876,0.038710,0.053277,0.044576
min,1.089290,1.090792,1.088489,1.081192,1.089921,1.105082,1.088902,1.105375,1.064337,1.059760,1.071716,1.076203,1.049627,1.048461,1.052063,1.054455,1.078883,1.061589,1.050131,1.106546,1.061300
25%,1.138896,1.141909,1.114315,1.120964,1.137774,1.135721,1.120496,1.123899,1.112026,1.108628,1.119213,1.108140,1.096656,1.101207,1.086736,1.084744,1.114130,1.111993,1.101736,1.158252,1.095646
50%,1.167029,1.164150,1.130415,1.131206,1.158025,1.151208,1.143578,1.137873,1.123804,1.123354,1.151841,1.131291,1.120403,1.112766,1.116891,1.108136,1.126153,1.121524,1.120277,1.199714,1.111041
75%,1.203717,1.210435,1.163417,1.160914,1.202950,1.195204,1.181250,1.176862,1.148574,1.150931,1.194279,1.169776,1.141042,1.131850,1.145949,1.140091,1.155170,1.145530,1.141626,1.241918,1.158180
max,1.303928,1.339175,1.215879,1.231460,1.273985,1.288133,1.275218,1.320151,1.228081,1.237506,1.284055,1.303694,1.230467,1.217953,1.264881,1.213991,1.203674,1.215333,1.216861,1.328495,1.241033


In [41]:
#ahora con dos tablas separadas la actividad y el reposo, 
# haré una nueva columna con los sujetos asi puedo hacerlos coincidir luego.


#df_reposo

In [42]:
df_reposo["nombre"] = df_reposo.index.str[:9]

C:\Users\yamal\AppData\Local\Temp\ipykernel_59824\1744317788.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_reposo["nombre"] = df_reposo.index.str[:9]


In [43]:
df_actividad["nombre"] = df_actividad.index.str[:9]

C:\Users\yamal\AppData\Local\Temp\ipykernel_59824\2854395664.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_actividad["nombre"] = df_actividad.index.str[:9]


In [44]:
df_reposo = df_reposo.set_index("nombre")
df_actividad = df_actividad.set_index("nombre")


In [45]:
df_variacion = df_actividad - df_reposo

In [46]:
df_variacion

,EEG Fp1,EEG Fp2,EEG F3,EEG F4,EEG F7,EEG F8,EEG T3,EEG T4,EEG C3,EEG C4,EEG T5,EEG T6,EEG P3,EEG P4,EEG O1,EEG O2,EEG Fz,EEG Cz,EEG Pz,EEG A2-A1,ECG ECG
nombre,,,,,,,,,,,,,,,,,,,,,
Subject00,0.032063,0.011210,0.009390,0.007432,0.019669,0.023425,0.018098,0.018563,0.006437,0.006838,0.025447,0.020929,0.022374,0.023006,0.012471,0.011723,0.006682,0.004271,0.025778,0.075302,0.014865
Subject01,0.030775,0.022563,-0.004017,0.013390,0.004257,0.019472,-0.022871,0.015975,-0.009610,0.013635,-0.033652,0.013378,-0.003713,0.019496,0.028907,0.021124,0.001529,0.004742,0.036677,0.009733,0.088939
Subject02,0.007227,0.044106,0.028991,-0.025836,0.013400,-0.005878,0.049514,0.037043,0.013785,0.002631,0.067947,0.073694,0.004022,0.004200,0.006736,0.010253,0.018808,0.004314,-0.001030,0.005543,-0.048818
Subject03,0.024126,0.037278,0.019953,0.015661,0.008166,0.018274,0.018051,0.016659,0.007774,0.008010,0.028066,0.010349,0.019982,0.030078,0.025121,0.022710,0.021732,0.027888,0.010547,0.017922,0.021802
Subject04,0.001175,-0.018716,-0.007635,-0.012420,0.007002,-0.013154,-0.014209,-0.006437,-0.014484,-0.024148,-0.005006,-0.005450,-0.014766,-0.006950,-0.007248,-0.003982,-0.015764,-0.019005,-0.005310,0.019372,-0.054055
Subject05,0.056074,-0.010725,0.014279,0.025659,0.033172,0.007739,0.016947,0.037115,0.025419,0.029192,0.006155,0.019036,0.027962,0.025124,0.005590,0.001920,0.023817,0.036277,0.025271,0.085842,-0.005216
Subject06,0.056300,0.072474,0.068204,0.049531,0.058380,0.062429,0.085581,0.094203,0.070438,0.084120,0.114659,0.085365,0.089087,0.104303,0.064529,0.073210,0.052747,0.071652,0.091839,0.032031,0.020883
Subject07,-0.013290,0.024615,0.036334,0.044117,0.021062,0.077172,0.048009,0.057366,0.036103,0.033343,0.039138,0.037326,0.031150,0.037663,0.023793,0.030659,0.035454,0.031869,0.037131,0.002672,0.014864
Subject08,-0.001662,-0.035310,0.014718,0.013436,0.017227,-0.007072,0.023969,0.023250,0.019242,0.015757,0.021483,0.024276,0.024223,0.016978,0.022277,0.015785,0.014041,0.012872,0.021238,-0.072344,-0.003112


In [47]:
df_variacion.describe()

,EEG Fp1,EEG Fp2,EEG F3,EEG F4,EEG F7,EEG F8,EEG T3,EEG T4,EEG C3,EEG C4,EEG T5,EEG T6,EEG P3,EEG P4,EEG O1,EEG O2,EEG Fz,EEG Cz,EEG Pz,EEG A2-A1,ECG ECG
count,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000
mean,0.027573,0.025818,0.019981,0.019029,0.022957,0.022559,0.026879,0.030315,0.018313,0.019676,0.028057,0.025669,0.022345,0.024252,0.025712,0.022066,0.016518,0.017925,0.024854,0.030697,0.008398
std,0.035353,0.040958,0.025407,0.026864,0.032088,0.042716,0.034267,0.041188,0.024600,0.032841,0.034204,0.039140,0.029489,0.029369,0.030438,0.031742,0.021399,0.023392,0.027351,0.047123,0.030134
min,-0.044077,-0.061408,-0.044381,-0.055114,-0.038035,-0.089277,-0.044793,-0.061321,-0.040254,-0.095470,-0.058182,-0.118492,-0.062766,-0.036679,-0.046393,-0.076754,-0.039817,-0.047076,-0.035176,-0.072344,-0.079744
25%,0.004648,0.001399,0.006829,0.008666,0.006740,-0.003599,0.004135,0.011952,0.006619,0.007717,0.005449,0.012638,0.010884,0.014294,0.006840,0.008787,0.006957,0.004631,0.010456,0.004909,-0.002597
50%,0.023988,0.023354,0.013331,0.017650,0.018791,0.014713,0.021592,0.024366,0.014057,0.014696,0.027615,0.020977,0.023595,0.024358,0.027517,0.021222,0.015116,0.013050,0.025853,0.029724,0.008175
75%,0.056130,0.044597,0.038632,0.031479,0.040228,0.046132,0.046715,0.046565,0.035149,0.033692,0.051656,0.045687,0.037200,0.035908,0.048254,0.040595,0.028299,0.032960,0.037270,0.064605,0.016795
max,0.108779,0.117008,0.085482,0.093426,0.111075,0.157623,0.140412,0.167643,0.075953,0.084120,0.114659,0.119168,0.089087,0.104303,0.073260,0.073210,0.063055,0.071652,0.091839,0.137331,0.088939


In [51]:
info_sujetos.head()

,Subject,Age,Gender,Number of subtractions,Count quality
0,Subject00,21,F,9.70,0
1,Subject01,18,F,29.35,1
2,Subject02,19,F,12.88,1
3,Subject03,17,F,31.00,1
4,Subject04,17,F,8.60,0


Ahora ya tenemos la variacion de la dimencion fractal entre estado reposo y resolución de ejercicios y tenemos ademas dos grupos de sujetos: los que aprobaron y los que no. Lo mas conveniente es usar una herramienta estadistica para comparar dos grupos independientes: prueba U de Mann–Whitney. Python tiene esta función.


Pruebo con Fp1 la prueba U

In [58]:
# Unimos las tablas por el nombre del sujeto
df = info_sujetos.merge(df_variacion, left_on='Subject', right_index=True)

# Separar ΔDF de Fp1 según grupo
grupo0 = df[df['Count quality'] == 0]['EEG Fp1']
grupo1 = df[df['Count quality'] == 1]['EEG Fp1']

In [59]:
u_stat, p_value = mannwhitneyu(grupo0, grupo1, alternative='two-sided')

In [60]:
print("Grupo 0 (no aprobaron) ΔDF Fp1:", grupo0.values)
print("Grupo 1 (aprobaron) ΔDF Fp1:", grupo1.values)
print("U-statistic:", u_stat)
print("p-value:", p_value)

Grupo 0 (no aprobaron) ΔDF Fp1: [ 0.03206269  0.00117512  0.05629986  0.02445592  0.02153204 -0.04407701
  0.07643624  0.01664593  0.01384621  0.00519272]
Grupo 1 (aprobaron) ΔDF Fp1: [ 0.03077472  0.00722701  0.02412571  0.05607391 -0.01328974 -0.00166182
  0.10877911  0.02880641 -0.0405826   0.01807707  0.0802482  -0.02173634
  0.02124354  0.06603761  0.0030157   0.0461944  -0.02091661  0.00281338
  0.07163359  0.04252301  0.07541165  0.0147205   0.06248692  0.02385043
  0.02839305  0.07480009]
U-statistic: 112.0
p-value: 0.5365259402507443


In [61]:
print("Media grupo0:", grupo0.mean())
print("Media grupo1:", grupo1.mean())

Media grupo0: 0.020356971707395276
Media grupo1: 0.03034803574076966


El cambio en Fp1 no parece depender del desempeño en matemática, según tus datos actuales. Ahora en todos los canales:

In [ ]:
canales = df_variacion.columns


resultados = {}


for canal in canales:
    grupo0 = df[df['Count quality'] == 0][canal]
    grupo1 = df[df['Count quality'] == 1][canal]
    

    u_stat, p_value = mannwhitneyu(grupo0, grupo1, alternative='two-sided')
    
   
    resultados[canal] = {
        'U-statistic': u_stat,
        'p-value': p_value,
        'mean_grupo0': grupo0.mean(),
        'mean_grupo1': grupo1.mean()
    }


df_resultados = pd.DataFrame(resultados).T
print(df_resultados)

           U-statistic   p-value  mean_grupo0  mean_grupo1
EEG Fp1          112.0  0.536526     0.020357     0.030348
EEG Fp2          112.0  0.536526     0.024772     0.026220
EEG F3           110.0  0.491004     0.018599     0.020512
EEG F4           117.0  0.658864     0.019226     0.018953
EEG F7            81.0  0.086722     0.009571     0.028105
EEG F8           107.0  0.426807     0.023668     0.022132
EEG T3           108.0  0.447644     0.022548     0.028545
EEG T4           119.0  0.710753     0.036918     0.027775
EEG C3           115.0  0.608568     0.016994     0.018820
EEG C4           115.0  0.608568     0.020813     0.019239
EEG T5           114.0  0.584078     0.028086     0.028046
EEG T6           102.0  0.331419     0.022199     0.027004
EEG P3           121.0  0.764019     0.022199     0.022401
EEG P4           114.0  0.584078     0.023971     0.024360
EEG O1           116.0  0.633504     0.021792     0.027220
EEG O2           122.0  0.791095     0.021452     0.0223

ningún canal muestra diferencia estadísticamente significativa entre los grupos.